# Stock Market Trend Classification on Historical Data
**NIELIT Chennai – PG Program in Data Engineering**

Goal: classify whether a stock will go **UP or DOWN** after 1 day / 1 week / 1 month, using technical indicators from ~20 years of daily data of 40 NSE stocks (data up to today).

> Run `python download_data.py` once (needs internet) so that `data/stock_data.csv` exists, then run this notebook top to bottom.

## Step 1 – Imports

In [ ]:
import json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
from features import HORIZONS, FEATURE_COLUMNS, build_dataset, time_split
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
import os; os.makedirs('models', exist_ok=True); os.makedirs('reports', exist_ok=True)

## Step 2 – Load raw data

In [ ]:
raw = pd.read_csv('data/stock_data.csv', parse_dates=['Date'])
print('Rows:', len(raw), '| Stocks:', raw.Ticker.nunique())
print('Date range:', raw.Date.min().date(), '->', raw.Date.max().date())
raw.head()

In [ ]:
raw.groupby('Ticker').agg(rows=('Close','size'), first=('Date','min'), last=('Date','max')).T

## Step 3 – Data quality checks

In [ ]:
print('Missing values:\n', raw.isna().sum())
print('Duplicate (Ticker, Date):', raw.duplicated(['Ticker','Date']).sum())
print('Non-positive prices:', (raw[['Open','High','Low','Close']] <= 0).sum().sum())

## Step 4 – Exploratory analysis

In [ ]:
tickers = raw.Ticker.unique()[:4]
fig, ax = plt.subplots(2, 2, figsize=(13, 7))
for a, t in zip(ax.ravel(), tickers):
    g = raw[raw.Ticker == t]; a.plot(g.Date, g.Close); a.set_title(t + ' – Close price')
plt.tight_layout(); plt.savefig('reports/price_history.png', dpi=120); plt.show()

In [ ]:
raw = raw.sort_values(['Ticker','Date'])
raw['daily_ret'] = raw.groupby('Ticker').Close.pct_change()
raw['daily_ret'].clip(-0.15, 0.15).hist(bins=100, figsize=(8,4)); plt.title('Distribution of daily returns')
plt.savefig('reports/return_distribution.png', dpi=120); plt.show()
print('Share of UP days:', round((raw.daily_ret > 0).mean(), 3))

## Step 5 – Feature engineering + target creation
Every indicator uses **only past data** (no peeking). Targets: `1` if the close after *h* trading days is higher than today's close, else `0`.

In [ ]:
data = build_dataset(raw[['Date','Open','High','Low','Close','Volume','Ticker']])
print('Model-ready rows:', len(data), '| Features:', len(FEATURE_COLUMNS))
data[['Date','Ticker','Close'] + FEATURE_COLUMNS[:6] + ['target_1D','target_1W','target_1M']].tail()

In [ ]:
for k in HORIZONS: print(k, 'share UP =', round(data['target_'+k].mean(), 3))

In [ ]:
plt.figure(figsize=(12, 9)); sns.heatmap(data[FEATURE_COLUMNS].corr(), cmap='coolwarm', center=0)
plt.title('Feature correlation'); plt.savefig('reports/feature_correlation.png', dpi=120, bbox_inches='tight'); plt.show()

## Step 6 – Time-based split (train → validation → test)
We split by **date**, never randomly (random splits leak the future into training). A purge gap removes samples whose label window overlaps the next set.

In [ ]:
splits = {}
for k, h in HORIZONS.items():
    tr, va, te = time_split(data, 'target_'+k, h)
    splits[k] = (tr, va, te)
    print(f'{k}: train {len(tr):,} ({tr.Date.min().date()}–{tr.Date.max().date()}) | val {len(va):,} | test {len(te):,} ({te.Date.min().date()}–{te.Date.max().date()})')

## Step 7 – Train and compare models
Three models per horizon. Selection is by **validation AUC** (the test set is untouched until the end). The baseline is always predicting the most common class.

In [ ]:
def make_models():
    return {
        'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=0.1)),
        'Random Forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=50, max_features='sqrt', n_jobs=-1, random_state=42),
        'Hist Gradient Boosting': HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=250, l2_regularization=1.0, random_state=42),
    }
val_rows, fitted = [], {}
for k in HORIZONS:
    tr, va, te = splits[k]; t = 'target_'+k
    for name, m in make_models().items():
        m.fit(tr[FEATURE_COLUMNS], tr[t])
        p = m.predict_proba(va[FEATURE_COLUMNS])[:, 1]
        val_rows.append(dict(Horizon=k, Model=name, Val_AUC=roc_auc_score(va[t], p), Val_Accuracy=accuracy_score(va[t], p > 0.5)))
val_df = pd.DataFrame(val_rows); val_df.round(4)

In [ ]:
best = val_df.loc[val_df.groupby('Horizon').Val_AUC.idxmax()].set_index('Horizon'); best.round(4)

## Step 8 – Final test evaluation
Re-train the chosen model on train + validation, then evaluate **once** on the untouched test period.

In [ ]:
results, models_meta, test_preds = [], {}, {}
for k in HORIZONS:
    tr, va, te = splits[k]; t = 'target_'+k
    name = best.loc[k, 'Model']
    model = make_models()[name]
    trva = pd.concat([tr, va])
    model.fit(trva[FEATURE_COLUMNS], trva[t])
    p = model.predict_proba(te[FEATURE_COLUMNS])[:, 1]; pred = (p > 0.5).astype(int)
    baseline = max(te[t].mean(), 1 - te[t].mean())
    r = dict(Horizon=k, Model=name, Accuracy=accuracy_score(te[t], pred), Baseline=baseline,
             AUC=roc_auc_score(te[t], p), Precision=precision_score(te[t], pred), Recall=recall_score(te[t], pred), F1=f1_score(te[t], pred))
    results.append(r); test_preds[k] = (te[t].values, pred, p)
    joblib.dump(model, f'models/model_{k}.joblib'); models_meta[k] = r
res_df = pd.DataFrame(results).set_index('Horizon'); res_df.round(4)

In [ ]:
print('Accuracy beats the majority-class baseline only if Accuracy > Baseline; AUC 0.5 = coin flip.')
for k,(y,pred,p) in test_preds.items():
    print('\n=====', k, '====='); print(classification_report(y, pred, target_names=['DOWN','UP']))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (k,(y,pred,p)) in zip(ax, test_preds.items()):
    sns.heatmap(confusion_matrix(y, pred), annot=True, fmt='d', cmap='Blues', ax=a, xticklabels=['DOWN','UP'], yticklabels=['DOWN','UP'])
    a.set_title(f'{k} horizon'); a.set_xlabel('Predicted'); a.set_ylabel('Actual')
plt.tight_layout(); plt.savefig('reports/confusion_matrices.png', dpi=120); plt.show()

## Step 9 – Which features matter? (permutation importance, 1-week model)

In [ ]:
tr, va, te = splits['1W']; m = joblib.load('models/model_1W.joblib')
samp = te.sample(min(5000, len(te)), random_state=1)
pi = permutation_importance(m, samp[FEATURE_COLUMNS], samp['target_1W'], scoring='roc_auc', n_repeats=5, random_state=1, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATURE_COLUMNS).sort_values()
imp.tail(15).plot.barh(figsize=(7, 5)); plt.title('Top 15 features (drop in AUC when shuffled)')
plt.tight_layout(); plt.savefig('reports/feature_importance.png', dpi=120); plt.show()

## Step 10 – Save metadata for the web app

In [ ]:
meta = dict(feature_columns=FEATURE_COLUMNS, horizons=HORIZONS, tickers=sorted(raw.Ticker.unique().tolist()),
            data_last_date=str(raw.Date.max().date()), n_rows=int(len(data)), metrics=models_meta)
json.dump(meta, open('models/metadata.json', 'w'), indent=2, default=float)
print('Saved models/ and metadata.json. Now run:  streamlit run app.py')

## Conclusions (write your own after running)
- Compare Accuracy vs Baseline and AUC for each horizon.
- Stock direction is very noisy: an AUC of about 0.50–0.58 is normal and honest. Be suspicious of anything above ~0.65 (usually data leakage).
- Limitations: only technical indicators, no news/fundamentals, past patterns may not repeat.
- Future work: add NIFTY index/market features, sentiment from news, LSTM comparison, walk-forward backtesting.